In [0]:
from pypdf import PdfReader
from pyspark.sql.types import * 
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from pyspark.sql import Row 
from pyspark.sql.functions import * 
from datetime import datetime
from pyspark.sql.window import Window 
import numpy as np 
from openai import OpenAI
import json 

In [0]:
%pip install openai
%pip install pypdf 
%pip install langchain
%pip install langchain-text-splitters


In [0]:
%restart_python 

In [0]:
[f.name for f in dbutils.fs.ls("/Volumes/retail_data/bronze/raw_data_volume/")]

['Abhishek Nandankar Latest CV.pdf', 'support_tickets.csv']

In [0]:
api_key ="sk-proj******"

In [0]:


path = "/Volumes/retail_data/bronze/raw_data_volume/Abhishek Nandankar Latest CV.pdf"

reader = PdfReader(path)

# print("total pages" , len(reader.pages))

text = ""
rows = []

for page_no ,page   in enumerate(reader.pages , start = 1) :
    text = page.extract_text() or ""

    rows.append(
        Row(
        document_name = "Abhishek_nandankar_cv"  , 
        page_number = page_no , 
        page_text = text)
        )

df_pdf = spark.createDataFrame(rows)

df_pdf = df_pdf.withColumn("Time_Stamp" , current_timestamp())  ## withColumn("index"  , monotonically_increasing_id())



In [0]:
%skip
def chunk_text(text , chunk_size = 500 , chunk_overlap = 100) :

    chunk = []
    start = 0 

    while start < len(text) :
        end = start + chunk_size 
        chunk.append(text[start:end])

        start += chunk_size - chunk_overlap 

    return chunk

In [0]:
%skip
text = df_pdf.select("page_text").collect()[0].page_text

pdf_text = chunk_text(text , chunk_size = 400 , chunk_overlap = 100)

chunk_rows = []
for chunk_no , chunk in enumerate(pdf_text , start = 1) :

    chunk_rows.append(
        Row(Doc_name = "Abhishek_Nandankar_cv" , 
            chunk = chunk_no , 
            page_number = 1 , 
            chunk_text = chunk)
    )

df_chunks = spark.createDataFrame(chunk_rows)

df_chunks = df_chunks.withColumn("DOC_id" , monotonically_increasing_id()).withColumn("Time_Stamp" , current_timestamp())

# df_chunks.display()

# for i , chunk in enumerate(pdf_text , start = 1) :
#     print(f"\n Chunk {i}")
#     print(chunk)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7490744780277326>, line 3
      1 text = df_pdf.select("page_text").collect()[0].page_text
----> 3 pdf_text = chunk_text(text , chunk_size = 400 , chunk_overlap = 100)
      5 chunk_rows = []
      6 for chunk_no , chunk in enumerate(pdf_text , start = 1) :

NameError: name 'chunk_text' is not defined

In [0]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 400 , chunk_overlap = 80 , separators = ["\n\n" , "\n" , "." ,"!" , "?" , " "])

text = df_pdf.first().page_text

chunk_text = text_splitter.split_text(text)

chunks = []
for i , chunk  in enumerate(chunk_text , start = 1) :

    chunks.append(
        Row(
            Doc_Name = "Abhishek_nandankar_CV" ,
            Chunk_id = i ,
            chunk_text = chunk 
        )
    )

    print(f"\n chunk {i}")
    print(f"\n {chunk}")

df_chunk_langchain = spark.createDataFrame(chunks)

df_chunk_langchain.display()


 chunk 1

 Abhishek Nandankar 
Data Engineer 
+91 9766024289 | abhisheknandankar2@gmail.com |  
linkedin.com/in/abhishek-nandankar-784a07244 | github.com/abhishek29996 
 
PROFESSIONAL SUMMARY 
Databricks Certified Data Engineer Professional with 3.5+ years  of experience building scalable data pipelines and

 chunk 2

 analytics solutions on Databricks, PySpark, S QL and Azure in the fintech domain. Specialized in Spark optimization, 
distributed ETL workflows and Medallion architecture, improving pipeline execution time by up to 40%. Skilled in Delta 
Lake, dimensional modelling, data governance and AI -powered self-service analytics, with working knowledge of AWS

 chunk 3

 (S3, Glue, Athena, Lambda). Seeking Data Engineer roles on Azure / Databricks lakehouse platforms. 
 
TECHNICAL SKILLS 
• Languages & Query: Python, PySpark, Scala, SQL, Spark SQL, Pandas, Matplotlib. 
• Data Engineering & Modelling: ETL/ELT Pipelines, Data Ingestion & Integration, Data Lakehouse, Medallion

 ch

Doc_Name,Chunk_id,chunk_text
Abhishek_nandankar_CV,1,Abhishek Nandankar Data Engineer +91 9766024289 | abhisheknandankar2@gmail.com | linkedin.com/in/abhishek-nandankar-784a07244 | github.com/abhishek29996 PROFESSIONAL SUMMARY Databricks Certified Data Engineer Professional with 3.5+ years of experience building scalable data pipelines and
Abhishek_nandankar_CV,2,"analytics solutions on Databricks, PySpark, S QL and Azure in the fintech domain. Specialized in Spark optimization, distributed ETL workflows and Medallion architecture, improving pipeline execution time by up to 40%. Skilled in Delta Lake, dimensional modelling, data governance and AI -powered self-service analytics, with working knowledge of AWS"
Abhishek_nandankar_CV,3,"(S3, Glue, Athena, Lambda). Seeking Data Engineer roles on Azure / Databricks lakehouse platforms. TECHNICAL SKILLS • Languages & Query: Python, PySpark, Scala, SQL, Spark SQL, Pandas, Matplotlib. • Data Engineering & Modelling: ETL/ELT Pipelines, Data Ingestion & Integration, Data Lakehouse, Medallion"
Abhishek_nandankar_CV,4,"Architecture, Data Warehousing, Dimensional Modelling (Star Schema, SCD Type 1/2), Batch & Incremental Processing • Big Data & Compute: Apache Spark, Databricks (Delta Lake, Unity Catalog, Genie AI), Azure Event Hubs • Performance Optimization: Partitioning, Caching, Broadcast Joins, AQE, Z-Ordering, Salting, Query Optimization"
Abhishek_nandankar_CV,5,"• Azure: Azure Data Factory, Azure Databricks, Synapse Analytics, ADLS Gen2, Azure SQL, Azure Blob Storage • AWS: S3, Glue (Crawler, Data Catalog), Athena, Lambda, CloudWatch • Governance & BI: Data Governance, Data Quality & Validation, Data Security, Access Control, Tableau, Self-Service Analytics • DevOps & Version Control: Git, CI/CD, Azure DevOps CERTIFICATIONS & ACHIEVEMENTS"
Abhishek_nandankar_CV,6,"CERTIFICATIONS & ACHIEVEMENTS • Databricks Certified Data Engineer Professional: Valid Jul 2026 – Jul 2028 • Silver Award, Poonawalla Fincorp: Recognized for outstanding contribution to the RBI Data Indent & Tableau Migration project, for excellence in data pipeline delivery and reporting readiness. EXPERIENCE Poonawalla Fincorp — Data Engineer Pune, India | Jul 2023 – Sep 2026"
Abhishek_nandankar_CV,7,"Poonawalla Fincorp — Data Engineer Pune, India | Jul 2023 – Sep 2026 • Pipeline Migration: Cut pipeline execution time by 35–40% and doubled query throughput by leading the full migration of 35+ production SQL Server reports to a Databricks-based distributed processing environment. • Architecture & Ingestion: Designed an Azure Data Lakehouse on Medallion architecture (Raw/Curated/Consumption)"
Abhishek_nandankar_CV,8,"and built Azure Data Factory ETL/ELT pipelines ingesting structured and semi-structured data from REST APIs, Event Hubs streams, SQL databases and SFTP into ADLS Gen2, with zero SLA breaches. • Processing & Reliability: Processed millions of records daily with Azure Databricks, PySpark, Spark SQL and Delta"
Abhishek_nandankar_CV,9,"Lake, sustaining 99%+ pipeline reliability through proactive monitoring, alerting and incident resolution. • Delta Lake: Implemented ACID transactions, schema evolution, time travel and incremental data loads to improve data reliability. • Spark Optimization: Optimized PySpark/Spark workloads using partitioning, caching, broadcast joins, AQE, salting and"
Abhishek_nandankar_CV,10,"Z-Ordering, Liquid clustering, reducing execution time by up to 40%. • Modelling & Automation: Architected data warehouse models (fact/dimension, star schema, SCD Type 1/2) for historical reporting and automated multi-source ingestion and scheduled refreshes, reducing manual data prep effort by 60–70%."


In [0]:

#### here we are testing for first chunk only 

from openai import OpenAI
client = OpenAI(api_key = api_key)

#### Chunk Emdedding
chunk_embedding = df_chunk_langchain.first().chunk_text

chunk_response = client.embeddings.create(
    model = "text-embedding-3-small" , 
    input = chunk_embedding
)

chunk_embedding_response = chunk_response.data[0].embedding


### Query Embedding 
query_response = client.embeddings.create(
    model = "text-embedding-3-small" ,
    input = "which certfication is done by Abhishek ?"
)

query_embedding_response = query_response.data[0].embedding


### check embeddings 
print(f"Length of Chunk Embedding : {len(chunk_embedding_response)}")
print(f"Length of Query Embedding : {len(query_embedding_response)}")
print(f"\n Chunk Embedding Response : {chunk_embedding_response[:10]}")
print(f"\n Query Embedding Response : {query_embedding_response[:10]}")

print(f"\n Chunk Embedding Usage : {chunk_response.usage} \n Query Embedding Usage : {query_response.usage} \n Total_Usage : {chunk_response.usage.total_tokens + query_response.usage.total_tokens}")


#### Comparing the Cosine Similarity of the Chunk and Query Embedding

import numpy as np 

def cosine_similarity(embed_1 , embed_2) :
    
    similarity  =  np.dot(embed_1 , embed_2)/(np.linalg.norm(embed_1)*np.linalg.norm(embed_2))
    
    return f"cosine similarity : {similarity}"

print(cosine_similarity(chunk_embedding_response , query_embedding_response))


Length of Chunk Embedding : 1536
Length of Query Embedding : 1536

 Chunk Embedding Response : [0.00024890899658203125, -0.007598876953125, 0.055877685546875, -0.0224456787109375, -0.0264129638671875, 0.0175933837890625, -0.00806427001953125, 0.032318115234375, -0.053314208984375, -0.021942138671875]

 Query Embedding Response : [-0.01030731201171875, -0.003139495849609375, 0.072998046875, -0.01983642578125, -0.03131103515625, 0.00569915771484375, -0.01143646240234375, 0.054901123046875, -0.00885772705078125, -0.00624847412109375]

 Chunk Embedding Usage : Usage(prompt_tokens=85, total_tokens=85) 
 Query Embedding Usage : Usage(prompt_tokens=11, total_tokens=11) 
 Total_Usage : 96
cosine similarity : 0.4969106831623045


In [0]:
###   Function for Cosine Similarity

import numpy as np 
from openai import OpenAI

client = OpenAI(api_key = api_key)

def similarity(text):

    chunk_txt = RecursiveCharacterTextSplitter(chunk_size = 400 , chunk_overlap=100 , separators= ["\n\n","\n",",",".","?"]).split_text(text)

    for i , chunk in enumerate(chunk_txt , start = 1):

        chunk_response =  client.embeddings.create(
            model = "text-embedding-3-small" , 
            input = chunk
        )

        chunk_emb = chunk_response.data[0].embedding

        query_response = client.embeddings.create(
            model = "text-embedding-3-small" ,
            input = f"is the important key word for Databricks Data Engineering present in this  {chunk} "
        )

        query_emb = query_response.data[0].embedding

        similarity = np.dot(chunk_emb , query_emb)/(np.linalg.norm(chunk_emb)*np.linalg.norm(query_emb))


        print(f" \n chunk {i} \n {chunk} \n Chunk embedding usage : {chunk_response.usage.total_tokens}   Query Embedding usage : {query_response.usage.total_tokens} Total  :  {chunk_response.usage.total_tokens + query_response.usage.total_tokens} \n Cosine Similarity : {similarity}")


In [0]:
text = df_pdf.first().page_text

print(similarity(text))

In [0]:

path = "/Volumes/retail_data/bronze/raw_data_volume/Abhishek Nandankar Latest CV.pdf"

[f for f in path.split("/")][-1]

'Abhishek Nandankar Latest CV.pdf'

In [0]:
### Create Dataframe from PDF

client = OpenAI(api_key = api_key)

query = "Does the text have keyword required for Databricks Dataengineering ?"

def  get_embeddings(pdf) :

    read_pdf = PdfReader(pdf)

    text = "\n\n".join( page.extract_text() or "" for page in read_pdf.pages)
   
    pdf_chunks = RecursiveCharacterTextSplitter(chunk_size = 300 , chunk_overlap = 100 , separators = ["\n\n" , "\n" , "," , "." , "?" , " "]).split_text(text)

    chunk_pdf = []

    for i , chunk in enumerate(pdf_chunks , start = 1) :
        chunk_response = client.embeddings.create(
            model = "text-embedding-3-small" ,
            input = chunk
        )

        chunk_embeddings = chunk_response.data[0].embedding

        query_response  =  client.embeddings.create(
            model = "text-embedding-3-small" ,
            input = query 
        )

        query_embeddings = query_response.data[0].embedding

        Cosine_similarity = np.dot(chunk_embeddings , query_embeddings)/(np.linalg.norm(chunk_embeddings)*np.linalg.norm(query_embeddings))

        chunk_pdf.append(
            Row( 
                Document_Name = [f for f in pdf.split("/")][-1] , 
                chunk_id = i , 
                chunk_text = chunk , 
                chunk_embedding = chunk_embeddings , 
                query_embedding = query_embeddings , 
                Cosine_similarity = float(Cosine_similarity)
            )
        )

    return spark.createDataFrame(chunk_pdf)



pdf = "/Volumes/retail_data/bronze/raw_data_volume/Abhishek Nandankar Latest CV.pdf"

df_pdf_cosine = get_embeddings(pdf)

df_pdf_cosine.display()

In [0]:
top_5_chunk =  df_pdf_cosine.select("chunk_id" , "chunk_text" , "Cosine_similarity").orderBy(col("Cosine_similarity").desc()).limit(5)

rows = top_5_chunk.collect()

context = ""

for row in rows :
    context += row["chunk_text"] + "\n\n"

# print(context)

# context = "\n\n".join(
#     row["chunk_text"] for row in rows
# )


question = "what is my experience with Databricks ?"

prompt = f"""
        You are a resume assistant.
        
        Answer the user's question using only the resume context provided below.

        If the answer is not avaliable in the context , clearly say that the information is not available . 

        Resume_context : {context}

        Question : {question}

        Answer : 
"""

response = client.responses.create(
    model = "gpt-5" ,
    input = prompt 
)

answer  =  response.output_text

if answer and answer.strip() :
    print(f"Final Answer : {answer}")
else :
    print("No Answer Found")

Final Answer : - Databricks Certified Data Engineer Professional (valid Jul 2026 – Jul 2028)
- 3.5+ years of data engineering experience, including hands-on work with Databricks
- Practical experience with Databricks components: Delta Lake, Unity Catalog, and Genie AI
- Deployed Databricks Genie for natural-language self-service analytics, saving ~15 analyst hours per week


In [0]:
evaluation_prompt  = f"""
You are a Strict RAG evaluator.

Evaluate Whether the generated answer is supported by the retrieved Resume context . 

Resume Context : {context}

User Question : {question}

Generated Answer : {answer}

Return only the Valid answer in this JSON format:
{{
    "faithfulness": "PASS or FAIL",
    "unsupported_claims": [],
    "reason": "Explanation"
}}

Rules :
- PASS only if all factual claims are supported by context. 
- FAIL of any factual claim is supported or contradicted .
- Do not assume missing information is True .

"""

evaluate_response  = client.responses.create(
    model = "gpt-5" , 
    input = evaluation_prompt
)

eveluation_text = evaluate_response.output_text

print(eveluation_text)


{
    "faithfulness": "PASS",
    "unsupported_claims": [],
    "reason": "All claims in the generated answer are directly supported by the resume context, including the Databricks certification validity period, 3.5+ years of experience with Databricks, practical experience with Delta Lake/Unity Catalog/Genie AI, and deploying Databricks Genie saving ~15 analyst hours per week."
}


In [0]:
max_similarity = df_pdf_cosine.agg(max("Cosine_similarity").alias("Max_Similarity")).collect()[0][0]

# print(max_similarity)

df_pdf_cosine.filter(col("Cosine_similarity") == max_similarity).display()

Document_Name chunk_id chunk_text chunk_embedding query_embedding Cosine_similarity Abhishek Nandankar Latest CV.pdf 5 Architecture, Data Warehousing, Dimensional Modelling (Star Schema, SCD Type 1/2), Batch & Incremental Processing 
• Big Data & Compute: Apache Spark, Databricks (Delta Lake, Unity Catalog, Genie AI), Azure Event Hubs List(-0.026275634765625, 0.011199951171875, 0.08203125, -0.00946044921875, 0.040771484375, -0.01102447509765625, -0.0081634521484375, 0.01389312744140625, -0.059326171875, 0.005947113037109375, 0.03448486328125, -0.026336669921875, -0.028350830078125, -0.005527496337890625, 0.0150146484375, 0.054656982421875, -0.0068206787109375, 0.0216217041015625, 0.0182037353515625, 0.028961181640625, 0.03326416015625, 0.03839111328125, -0.0012054443359375, -0.01092529296875, -0.00885772705078125, 0.03839111328125, -0.035797119140625, 0.046295166015625, 0.030548095703125, -0.04730224609375, 0.033721923828125, -0.0272674560546875, -0.01666259765625, 0.01123046875, 0.0075225830078125, 0.01325225830078125, -0.039093017578125, 0.058258056640625, 0.0265045166015625, 0.0183868408203125, 0.0120849609375, 0.00891876220703125, 0.0135040283203125, 0.03521728515625, 0.0157623291015625, 0.0148468017578125, -0.035369873046875, -0.0711669921875, -0.01200103759765625, 0.0307769775390625, -0.0200042724609375, -0.00543212890625, 0.02935791015625, -0.050933837890625, -0.0197601318359375, 0.00865936279296875, 0.0286102294921875, 0.011871337890625, 0.0268402099609375, -0.0250396728515625, 0.011810302734375, -0.019500732421875, -0.0017833709716796875, 0.014892578125, -0.03363037109375, 1.2159347534179688E-4, -0.03680419921875, -0.01486968994140625, -0.03369140625, -0.030029296875, -0.0264892578125, 0.0557861328125, 0.020904541015625, -0.01493072509765625, 0.0299530029296875, -0.0061187744140625, -0.040130615234375, 0.0234222412109375, -0.0218505859375, 0.017822265625, -0.0116119384765625, 0.0125579833984375, -0.00595855712890625, -0.0228118896484375, 0.004573822021484375, -0.021697998046875, 0.017974853515625, -0.00211334228515625, -0.06988525390625, 0.0012845993041992188, 0.0160675048828125, -0.0052947998046875, -0.00620269775390625, -0.015716552734375, 0.0199127197265625, 0.0193023681640625, 0.0543212890625, 0.0030307769775390625, -0.002819061279296875, 0.050048828125, -0.0148468017578125, -0.043548583984375, 0.027191162109375, -0.01160430908203125, 0.0304412841796875, -0.04119873046875, 0.0206146240234375, -0.0243072509765625, -0.031280517578125, -0.00931549072265625, -0.1466064453125, -0.054046630859375, -0.02899169921875, 0.0167694091796875, -0.045928955078125, 0.0118560791015625, 0.032257080078125, -0.0197906494140625, -0.0020542144775390625, -0.0118560791015625, -0.025177001953125, 0.056976318359375, -0.0215301513671875, -0.01061248779296875, -0.02056884765625, 0.00847625732421875, 0.007049560546875, -0.036865234375, 0.01216888427734375, 0.02099609375, 0.039276123046875, 0.007595062255859375, 0.059722900390625, -0.049835205078125, 0.043853759765625, -0.0015411376953125, -0.02740478515625, 0.0299224853515625, -0.0138092041015625, -0.027618408203125, 0.032135009765625, 0.01534271240234375, 0.03680419921875, 0.038909912109375, 0.0016107559204101562, -1.8084049224853516E-4, -0.0281982421875, 0.0391845703125, 0.023040771484375, -0.0165863037109375, -0.01404571533203125, -0.02935791015625, -0.0238189697265625, -0.0268402099609375, 0.019989013671875, -0.04150390625, 0.02178955078125, 0.02935791015625, -0.00489044189453125, -0.05731201171875, 0.033782958984375, 0.008819580078125, 0.012908935546875, 0.029144287109375, -0.017425537109375, -0.028411865234375, -0.0433349609375, -0.0010051727294921875, -0.056884765625, -0.0241851806640625, -0.03179931640625, -0.034637451171875, 0.0291748046875, -0.0037975311279296875, 0.029876708984375, 0.019134521484375, -0.016357421875, -0.018585205078125, 0.01294708251953125, -0.050323486328125, -0.04132080078125, 0.022186279296875, -0.033843994140625, -0.02313232421875, -0.002567291259765625, 0.0010

In [0]:
from openai import OpenAI

client = OpenAI(api_key= api_key)

response = client.responses.create(
    model="gpt-5",
    input= f"""which certification is done , share me name , ans validity of certification  :
    {chunk_ai}
    
    """
)

print(response.output_text)

- Certification name: Databricks Certified Data Engineer Professional
- Validity: 2 years from the date it was earned (exact expiry not determinable from the provided text). If you share the issue date, I can give the exact expiration date.


In [0]:
print(response.usage)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8493353481097482>, line 1
----> 1 print(response.usage)

NameError: name 'response' is not defined

In [0]:
dbutils.secrets.listScopes()

[]